## Sentiment Analysis

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("junaid6731/hospital-reviews-dataset")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import precision_score, recall_score, matthews_corrcoef

In [ ]:
import os

DATA_PATH = path

print(os.listdir(DATA_PATH))

In [ ]:
df = pd.read_csv(DATA_PATH + "/hospital.csv")
print(df.head())
print(df.shape)

In [ ]:
df = df.drop(columns=["Unnamed: 3"], errors="ignore")

In [ ]:
df = df.rename(columns={
    "Feedback": "text",
    "Sentiment Label": "label"
})

In [ ]:
df = df.dropna(subset=["text", "label"])

In [ ]:
print(df.head())
print(df["label"].value_counts())

In [ ]:
model_name = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

In [ ]:
dataset = Dataset.from_pandas(df)

In [ ]:
def tokenize(batch):

    return tokenizer(
        batch["text"],
        padding=True,
        truncation=True,
        max_length=128
    )


dataset = dataset.map(tokenize, batched=True)

In [ ]:
dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

In [ ]:
dataset = dataset.rename_column("label", "labels")

dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [ ]:
training_args = TrainingArguments(

    output_dir="./bert_patient_reviews",

    eval_strategy="epoch",

    per_device_train_batch_size=8,   # small dataset
    per_device_eval_batch_size=8,

    num_train_epochs=5,   # more epochs (small data)

    learning_rate=2e-5,

    logging_steps=20,

    save_strategy="epoch",

    load_best_model_at_end=True,

    report_to="none"
)

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=dataset["train"],
    eval_dataset=dataset["test"]
)

In [ ]:
trainer.train()

In [ ]:
# Already:
model.save_pretrained('sentimentmodeldir')
# Zip it:
!zip -r sentiment_model.zip sentimentmodeldir/
print("✅ BERT zipped!")


In [ ]:
preds = trainer.predict(dataset["test"])

y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=1)

In [ ]:
from sklearn.metrics import precision_score, recall_score, matthews_corrcoef, accuracy_score

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
mcc = matthews_corrcoef(y_true, y_pred)
accuracy = accuracy_score(y_true, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("MCC      :", mcc)

In [ ]:
def predict_sentiment(text):

    device = next(model.parameters()).device   # detect GPU/CPU

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # Move inputs to same device as model
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

        probs = torch.softmax(outputs.logits, dim=1)

    return probs.cpu().numpy()   # move back to CPU for printing

In [ ]:
review = "Doctors were careless and nurses were rude"

print(predict_sentiment(review))

In [ ]:
def predict_sentiment_label(text):

    probs = predict_sentiment(text)[0]

    if probs[1] > probs[0]:
        return "Positive Review", probs
    else:
        return "Negative Review", probs


print(predict_sentiment_label(review))

In [ ]:
save_path = "./clinicalbert_model"

tokenizer.save_pretrained(save_path)
model.save_pretrained(save_path)

**Pickling**